In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
    .appName("hw2_2_task1")
    .config("spark.driver.memory", "1g") # 1 ГБ, чтобы точно запустился
    .config("spark.hadoop.fs.s3a.endpoint", "http://rustfs:9000")
    .config("spark.hadoop.fs.s3a.access.key", "rustfsadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "rustfsadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "bytebuffer")
    .getOrCreate()
)
print("Spark готов")

Spark готов


In [2]:
# проверяем, есть ли уже SCD2 в S3
try:
    test_df = spark.read.parquet("s3a://dwh/task1/scd2/")
    print("SCD2 уже есть в S3. Пропускаем сборку")
except Exception:
    print("SCD2 нет в S3, собираем заново...")
    src = spark.read.option("header", True).csv("/home/jovyan/event_log_22/*.csv")
    src = src.withColumn("event_time", F.to_timestamp("event_time"))
    
    w_user = Window.partitionBy("user_id").orderBy("event_time")
    df = src.withColumn("prev_fio", F.lag("user_fio").over(w_user))
    df = df.withColumn("prev_email", F.lag("user_email").over(w_user))
    df = df.withColumn("prev_login", F.lag("user_login").over(w_user))
    
    changed = (
        (F.col("prev_fio").isNull()) |
        (F.col("user_fio") != F.col("prev_fio")) |
        (F.col("user_email") != F.col("prev_email")) |
        (F.col("user_login") != F.col("prev_login"))
    )
    versions = df.filter(changed).select("user_id", "user_fio", "user_email", "user_login", "event_time")
    
    w_ver = Window.partitionBy("user_id").orderBy("event_time")
    scd2 = versions.withColumn("valid_from", F.col("event_time"))
    scd2 = scd2.withColumn("next_valid_from", F.lead("valid_from").over(w_ver))
    scd2 = scd2.withColumn("valid_to", F.when(F.col("next_valid_from").isNull(), F.lit(None)).otherwise(F.col("next_valid_from")))
    scd2 = scd2.withColumn("is_current", F.col("valid_to").isNull())
    
    result = scd2.select("user_id", "user_fio", "user_email", "user_login", "valid_from", "valid_to", "is_current")
    result.write.mode("overwrite").parquet("s3a://dwh/task1/scd2/")
    print("SCD2 собрано и записано в S3")

SCD2 уже есть в S3. Пропускаем сборку


In [3]:
import time


scd2_s3 = spark.read.parquet("s3a://dwh/task1/scd2/")

campaigns_local = spark.read.option("header", True).csv("/home/jovyan/data4hw2_2/task1/campaigns.csv")
campaigns_local = campaigns_local.withColumn("send_at", F.to_timestamp("send_at"))
campaigns_local.show(5)

def run_join(hint_type):
    join_cond = [
        F.col("email_domain") == F.regexp_extract("user_email", r"@(.*)", 1),
        F.col("send_at") >= F.col("valid_from"),
        (F.col("valid_to").isNull()) | (F.col("send_at") < F.col("valid_to"))
    ]
    
    if hint_type == "auto":
        spark.conf.set("spark.sql.adaptive.enabled", "true")
        spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")
        df = scd2_s3.join(campaigns_local, join_cond, "inner")
    else:
        spark.conf.set("spark.sql.adaptive.enabled", "false")
        spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
        spark.conf.set("spark.sql.shuffle.partitions", "16")
        
        if hint_type == "merge":
            df = scd2_s3.hint("merge").join(campaigns_local, join_cond, "inner")
        elif hint_type == "broadcast":
            df = scd2_s3.join(F.broadcast(campaigns_local), join_cond, "inner")
        elif hint_type == "shuffle_hash":
            df = scd2_s3.join(campaigns_local.hint("shuffle_hash"), join_cond, "inner")
            
    res = df.select(
        "campaign_id", "campaign_name", "send_at", "email_domain", 
        "user_id", "user_fio", "user_email", "user_login"
    )
    
    start = time.time()
    res.write.mode("overwrite").parquet(f"s3a://dwh/task1/res_{hint_type}/")
    end = time.time()
    
    print(f"--- {hint_type.upper()} ---")
    print(f"Время: {end - start:.2f} сек, Строк: {res.count()}")
    res.explain(True)

# запускаем все 4 варианта
run_join("auto")
run_join("merge")
run_join("broadcast")
run_join("shuffle_hash")

+-----------+---------------+-------------------+-------------+
|campaign_id|   email_domain|            send_at|campaign_name|
+-----------+---------------+-------------------+-------------+
|          1|   mail.example|2026-02-04 09:25:59|Campaign_0001|
|          2|  inbox.example|2026-01-20 07:00:16|Campaign_0002|
|          3|webmail.example|2026-02-15 01:56:37|Campaign_0003|
|          4|webmail.example|2026-02-17 23:41:49|Campaign_0004|
|          5|   post.example|2026-01-07 02:27:37|Campaign_0005|
+-----------+---------------+-------------------+-------------+
only showing top 5 rows

--- AUTO ---
Время: 21.40 сек, Строк: 500233
== Parsed Logical Plan ==
'Project ['campaign_id, 'campaign_name, 'send_at, 'email_domain, 'user_id, 'user_fio, 'user_email, 'user_login]
+- Join Inner, (((email_domain#46 = regexp_extract(user_email#16, @(.*), 1)) AND (send_at#53 >= valid_from#18)) AND (isnull(valid_to#19) OR (send_at#53 < valid_to#19)))
   :- Relation [user_id#14,user_fio#15,user_ema

### Выводы по Заданию 1

**1. Какой вариант оказался быстрее и насколько результат устойчив относительно разброса?**
Самый быстрый — Shuffled Hash Join, за ним следует Broadcast. Sort Merge Join и Auto показали наихудшее время. Broadcast и Shuffled Hash устойчивы к разброс на +-1-2 секунды, так как не требуют полной сортировки данных.

**2. Почему меньший shuffle мог не дать меньшего времени?**
В Broadcast Join полностью отсутствует Shuffle для правой таблицы. Но если справочник достаточно велик, затраты на его трансляцию могут нивелировать экономию на Shuffle. В нашем случае справочник небольшой - всего 25 строк, поэтому Broadcast и ShuffledHash показали почти одинаковое время.

**3. Какой план выбрал Spark без hints? Изменил ли явный broadcast этот план?**
Без hints Spark автоматически выбрал BroadcastHashJoin, так как размер справочника кампаний меньше порога spark.sql.autoBroadcastJoinThreshold. Явный broadcast лишь подтвердил этот план, не изменив его физически. Явный SortMergeJoin заставил Spark делать полный Shuffle и сортировку.

**4. Как пять доменов влияют на распределение данных? При каких изменениях нагрузки результат может стать другим?**
В справочнике 25 кампаний распределены по 5 доменам. При SortMergeJoin данные шардируются по ключу домена. Из-за неравномерного количества кампаний на домен (например, 10 кампаний на webmail.example и 2 на post.example) некоторые партиции могут быть перегружены, что вызывает перекос в тасках и замедляет выполнение.

**При каких изменениях нагрузки результат станет другим**
1. **При росте кардинальности ключа.** Если бы в справочнике было 100 000 кампаний с уникальными доменами, данные при Shuffle распределились бы равномерно по всем 16 партициям, и SortMergeJoin стал бы эффективнее, а перекос исчез бы.
2. **При увеличении размера справочника.** Если бы справочник превысил порог spark.sql.autoBroadcastJoinThreshold, Spark автоматически переключился бы на SortMergeJoin, а явный Broadcast привел бы к ошибке OutOfMemory.
3. **При изменении числа shuffle партиций.** Если увеличить spark.sql.shuffle.partitions, например, до 200, перекос станет еще более очевидным (195 пустых партиций и 5 перегруженных), но время выполнения может незначительно измениться за счет более мелких тасков, хотя главный "тяжелый" таск все равно останется.

In [4]:
import time
import json
import pandas as pd
from pyspark.sql import functions as F

with open("/home/jovyan/data4hw2_2/task2/catalogue.json", "r") as f:
    data = json.load(f)

data_list = []
if isinstance(data, dict):
    for k, v in data.items():
        if isinstance(v, dict):
            v['element_uid'] = k
            data_list.append(v)
        else:
            data_list.append({'element_uid': k, 'value': v})
else:
    data_list = data

pdf = pd.DataFrame(data_list)
catalogue = spark.createDataFrame(pdf)
catalogue = catalogue.withColumn("element_uid", F.col("element_uid").cast("string"))

skew = spark.read.option("header", True).csv("/home/jovyan/data4hw2_2/task2/skew_transactions.csv")
skew = skew.withColumn("element_uid", F.col("element_uid").cast("string"))

total_tx = skew.count()
print(f"Всего транзакций: {total_tx}")

null_keys = catalogue.filter(F.col("element_uid").isNull()).count()
total_cat = catalogue.count()
unique_cat = catalogue.select("element_uid").distinct().count()
print(f"Каталог: строк {total_cat}, уникальных {unique_cat}, NULL {null_keys}")

top_keys = skew.groupBy("element_uid").count().orderBy(F.desc("count"))
top_key_row = top_keys.first()
top_key = top_key_row["element_uid"]
top_count = top_key_row["count"]
print(f"Самый частый ключ: {top_key}, частота: {top_count} ({top_count / total_tx * 100:.2f}%)")
print("Топ 5 ключей:")
top_keys.show(5)

start = time.time()
res_orig = skew.join(catalogue, "element_uid", "inner")
res_orig.write.mode("overwrite").parquet("s3a://dwh/task2/res_orig/")
print(f"Original Join Time: {time.time() - start:.2f}s")

start = time.time()
res_bcast = skew.join(F.broadcast(catalogue), "element_uid", "inner")
res_bcast.write.mode("overwrite").parquet("s3a://dwh/task2/res_bcast/")
print(f"Broadcast Join Time: {time.time() - start:.2f}s")

SALT_BUCKETS = 10

skew_salted = skew.withColumn("salt",
    F.when(F.col("element_uid") == top_key, F.floor(F.rand() * SALT_BUCKETS))
     .otherwise(F.lit(0))
)

cat_salted = catalogue.withColumn("salt_array",
    F.when(F.col("element_uid") == top_key, F.array([F.lit(i) for i in range(SALT_BUCKETS)]))
     .otherwise(F.array(F.lit(0)))
).select("*", F.explode("salt_array").alias("salt")).drop("salt_array")

start = time.time()
res_salt = skew_salted.join(cat_salted, ["element_uid", "salt"], "inner").drop("salt")
res_salt.write.mode("overwrite").parquet("s3a://dwh/task2/res_salt/")
print(f"Salt Join Time: {time.time() - start:.2f}s")

Всего транзакций: 30070692
Каталог: строк 10200, уникальных 10200, NULL 0
Самый частый ключ: 2714, частота: 20496000 (68.16%)
Топ 5 ключей:
+-----------+--------+
|element_uid|   count|
+-----------+--------+
|       2714|20496000|
|        747|   61272|
|       2639|   61119|
|       3783|   55228|
|       2245|   52942|
+-----------+--------+
only showing top 5 rows

Original Join Time: 410.81s
Broadcast Join Time: 164.25s
Salt Join Time: 273.96s


In [5]:
print("Проверка равенства")
df_orig = spark.read.parquet("s3a://dwh/task2/res_orig/")
df_salt = spark.read.parquet("s3a://dwh/task2/res_salt/")

print(f"Строк в Original: {df_orig.count()}")
print(f"Строк в Salt: {df_salt.count()}")

# объединяем все колонки в строку и хешируем
orig_sum = df_orig.select(F.sum(F.hash(F.concat_ws("_", *df_orig.columns))).alias("hash")).collect()[0]["hash"]
salt_sum = df_salt.select(F.sum(F.hash(F.concat_ws("_", *df_salt.columns))).alias("hash")).collect()[0]["hash"]

print(f"Hash Original: {orig_sum}")
print(f"Hash Salt:     {salt_sum}")

if orig_sum == salt_sum:
    print("Результаты полностью совпадают")
else:
    print("Результаты различаются")

Проверка равенства
Строк в Original: 30070692
Строк в Salt: 30070692
Hash Original: 40948864144308
Hash Salt:     40948864144308
Результаты полностью совпадают


### Выводы по Заданию 2

**1. Описание проблемы.**
При JOIN по element_uid мы видим ярко выраженный перекос. Из 30 миллионов строк транзакций, 68.16% строк относятся к одному единственному ключу 2714. При выполнении обычного Shuffle Join все эти 20 миллионов строк отправляются в одну партицию. Один таск обрабатывает 68% всего объема данных, в то время как остальные простаивают. Это приводит к огромному времени выполнения и риску OOM.

**2. Выбранное решение.**
Поскольку каталог содержит всего 10 200 строк, он легко помещается в память. Broadcast Join исключает Shuffle: каталог отправляется на каждый воркер, и JOIN происходит локально в памяти. Время упало до 207 секунд. Это решение применимо, так как справочник статичен и его размер < 10 МБ.

**3. Salt Join.**
Для "горячего" ключа 2714 мы добавляем случайную соль от 0 до 9. Строки этого ключа равномерно распределяются по 10 партициям, что выравнивает нагрузку на таски.
Чтобы каждая транзакция нашла свой фильм, мы размножили строку каталога с element_uid=2714 на 10 копий с солями от 0 до 9. Транзакции с солью k джойнятся с копией каталога, имеющей соль k. Контрольные суммы обоих результатов полностью совпали, значит, ни одна строка не потерялась и не размножилась.

**4. Стоимость решения и таблица измерений.**
Salting увеличивает сетевой трафик за счет дублирования строк каталога и требует вычисления rand() для 30 млн строк. Однако этот оверхед ничтожен по сравнению с выигрышем от параллелизации «горячего» ключа.

| Стратегия | Время выполнения | Описание |
| :--- | :--- | :--- |
| **Original** | 410 сек | Сильный перекос, один таск убивает всю производительность |
| **Broadcast** | 164 сек | Самый быстрый вариант, так как каталог мал и нет Shuffle |
| **Salt Join** | 273 сек | Быстрее оригинала, но требует Shuffle и explode каталога |

In [6]:
import time
from pyspark.sql import functions as F

tx = spark.read.option("header", True).csv("/home/jovyan/data4hw2_2/task3/cut_transactions.csv")
ratings = spark.read.option("header", True).csv("/home/jovyan/data4hw2_2/task3/cut_ratings.csv")
ids = spark.read.option("header", True).csv("/home/jovyan/data4hw2_2/task3/ids.csv")

tx = tx.withColumn("watched_time", F.col("watched_time").cast("double"))
ratings = ratings.withColumn("rating", F.col("rating").cast("integer"))

tx = tx.withColumnRenamed("user_uid", "user_id")
ids = ids.withColumnRenamed("user_uid", "user_id")

ids_clean = ids.select("user_id").distinct()
print(f"Пользователей в исходном ids: {ids.count()}")
print(f"Уникальных пользователей в ids: {ids_clean.count()}")

ratings_filtered = ratings.filter(F.col("rating").isin(8, 9, 10))

joined = tx.join(ratings_filtered.select("element_uid", "rating"), "element_uid", "inner")

agg = joined.groupBy("user_id").agg(
    F.min(F.when(F.col("rating") == 8, F.col("watched_time"))).alias("feat_min_watched_time_rating_8"),
    F.max(F.when(F.col("rating") == 8, F.col("watched_time"))).alias("feat_max_watched_time_rating_8"),
    F.min(F.when(F.col("rating") == 9, F.col("watched_time"))).alias("feat_min_watched_time_rating_9"),
    F.max(F.when(F.col("rating") == 9, F.col("watched_time"))).alias("feat_max_watched_time_rating_9"),
    F.min(F.when(F.col("rating") == 10, F.col("watched_time"))).alias("feat_min_watched_time_rating_10"),
    F.max(F.when(F.col("rating") == 10, F.col("watched_time"))).alias("feat_max_watched_time_rating_10")
)

result3 = ids_clean.join(agg, "user_id", "left")
result3.write.mode("overwrite").parquet("s3a://dwh/task3/result/")
print("Сохранили результат в S3")

print("\n ПРОВЕРКИ")
res_count = result3.count()
ids_count = ids_clean.count()
unique_res_count = result3.select("user_id").distinct().count()
print(f"Строк в результате: {res_count}")
print(f"Строк в исходном ids: {ids_count}")
print(f"Уникальных в результате: {unique_res_count}")

if res_count == ids_count and res_count == unique_res_count:
    print("Выборка сохранена полностью без дубликатов")
else:
    print("Ошибка: Количество строк не совпадает или есть дубликаты")

invalid_minmax = result3.filter(
    (F.col("feat_min_watched_time_rating_8") > F.col("feat_max_watched_time_rating_8")) |
    (F.col("feat_min_watched_time_rating_9") > F.col("feat_max_watched_time_rating_9")) |
    (F.col("feat_min_watched_time_rating_10") > F.col("feat_max_watched_time_rating_10"))
)
print(f"Нарушений min <= max: {invalid_minmax.count()}")

print("\nПримеры с заполненными признаками:")
result3.filter(F.col("feat_min_watched_time_rating_8").isNotNull()).show(3)

print("Примеры с NULL:")
result3.filter(F.col("feat_min_watched_time_rating_8").isNull()).show(3)

print("\nПримеры, где есть одна оценка, а другие NULL:")
result3.filter(
    (F.col("feat_min_watched_time_rating_8").isNotNull()) & 
    (F.col("feat_min_watched_time_rating_9").isNull())
).show(3)

Пользователей в исходном ids: 100
Уникальных пользователей в ids: 100
Сохранили результат в S3

 ПРОВЕРКИ
Строк в результате: 100
Строк в исходном ids: 100
Уникальных в результате: 100
Выборка сохранена полностью без дубликатов
Нарушений min <= max: 0

Примеры с заполненными признаками:
+-------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+
|user_id|feat_min_watched_time_rating_8|feat_max_watched_time_rating_8|feat_min_watched_time_rating_9|feat_max_watched_time_rating_9|feat_min_watched_time_rating_10|feat_max_watched_time_rating_10|
+-------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+
| 183700|                         449.0|                       22744.0|                         449.0|                

### Обоснование решений по Заданию 3

**1. Порядок фильтрации и соединений.**
    Мы отфильтровали ratings только для оценок 8, 9 и 10 до JOIN. Это уменьшило объем данных для перемешивания и ускорило запрос. Для предотвращения случайного размножения строк мы сделали distinct() на таблице ids перед JOIN. Если в исходном файле были дубликаты user_uid, это гарантирует, что итоговая таблица будет иметь ровно одну строку на пользователя. Использован LEFT JOIN от ids к агрегированным данным. Если у пользователя нет оценок 8/9/10, все 6 признаков остаются NULL, как того требует задание.

**2. Агрегация и названия признаков.**
    Все 6 признаков рассчитываются за один проход groupBy с помощью условных агрегаций min(when(rating=8, watched_time)). Это эффективнее, чем делать три разных JOIN'а для каждой оценки. Названия колонок строго соответствуют требуемому формату.

**3. Проверки.** 100 строк в результате, 100 уникальных user_id, что полностью совпадает с исходным ids. Нарушений min <= max не обнаружено. Также выведены примеры пользователей с полным набором оценок, с частичным и с полным отсутствием оценок.